# Практична робота №8
## Дисципліна: «Аналіз даних та математичне моделювання»
### Тема: «Групування і зведені таблиці: groupby, агрегувальні функції, pivot_table, crosstab»

---

**Студент:** Хрустовський Павло  
**Група:** ІТ-42  
**Варіант:** 2  
**Місто:** Львів  
**Середньорічна температура:** 8.5 °C  
**Сезонна амплітуда:** 12 °C  

**Мета заняття:** побудувати довгий (tidy) набір даних за 4 роки (2021–2024) та 12 місяців для міста Львів і застосувати до нього інструменти групування й зведення даних: `groupby()`, `.agg()`, `pivot_table()`, `crosstab()` та `pivot()`.

## Генерація довгого (tidy) набору даних для Варіанта 2 (Львів)

Відповідно до умов Варіанта 2 підставляємо параметри:
- `city = "Львів"`
- `base_temp = 8.5`
- `amplitude = 12`
- `np.random.seed(42)` для повної відтворюваності результату

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)               # для відтворюваності результату
base_temp = 8.5                  # середньорічна температура для Варіанта 2 (Львів)
amplitude = 12                   # сезонна амплітуда для Варіанта 2
city = "Львів"                   # місто

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)   # пік влітку (місяць 7)
        noise = np.random.normal(0, 1.0)
        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": round(base_temp + seasonal + noise, 1),
        })

climate = pd.DataFrame(rows)   # 48 рядків: 4 роки x 12 місяців, довгий формат
print(f"Розмірність датасету: {climate.shape}")
climate.head(12)

Розмірність датасету: (48, 4)


,місто,рік,місяць,температура
0,Львів,2021,1,-3.0
1,Львів,2021,2,-2.0
2,Львів,2021,3,3.1
3,Львів,2021,4,10.0
4,Львів,2021,5,14.3
5,Львів,2021,6,18.7
6,Львів,2021,7,22.1
7,Львів,2021,8,19.7
8,Львів,2021,9,14.0
9,Львів,2021,10,9.0


## Завдання 1. groupby і agg за роками

**Завдання:** Використовуючи набір `climate` (48 рядків), згрупуйте дані за "рік" і виведіть для кожного року середню, мінімальну й максимальну температуру одним викликом `.agg()`. Письмово визначте, чи є в цих чотирьох роках помітний тренд (наприклад, потепління), чи коливання виглядають випадковими.

In [2]:
# Групування за роком з одночасним обчисленням середнього, мінімуму та максимуму
yearly_stats = climate.groupby("рік")["температура"].agg(["mean", "min", "max"])
yearly_stats.columns = ["середня_темп", "мін_темп", "макс_темп"]
yearly_stats

,середня_темп,мін_темп,макс_темп
рік,,,
2021,8.791667,-3.0,22.1
2022,7.916667,-3.8,19.6
2023,8.300000,-4.0,20.7
2024,8.166667,-3.9,20.4


### Письмовий висновок до Завдання 1:
- **Отримані значення середньої температури:**
  - 2021 рік: **8.79 °C** (мін: -3.0 °C, макс: 22.1 °C)
  - 2022 рік: **7.92 °C** (мін: -3.8 °C, макс: 19.6 °C)
  - 2023 рік: **8.30 °C** (мін: -4.0 °C, макс: 20.7 °C)
  - 2024 рік: **8.17 °C** (мін: -3.9 °C, макс: 20.4 °C)
- **Аналіз наявності тренду:**
  У ряді спостережень за 2021–2024 роки **помітного тренду (потепління чи похолодання) немає**. Середньорічні значення коливаються у вузькому діапазоні від 7.92 °C до 8.79 °C навколо теоретичного математичного сподівання базової температури для Львова (8.5 °C). Після максимуму у 2021 році спостерігається спад у 2022 році, помірне підвищення у 2023 і знову невеликий спад у 2024. Така динаміка має суто **випадковий характер**, зумовлений додаванням нормального шуму $\mathcal{N}(0, 1)$ до детермінованої гармонійної складової.

## Завдання 2. groupby і agg за місяцями

**Завдання:** Згрупуйте дані за "місяць" і для кожного місяця обчисліть середню температуру й стандартне відхилення за чотири роки одним `.agg()`. Визначте, у якому місяці розкид (`std`) між роками найбільший, і запишіть письмове припущення, чому саме цей місяць виявився найнестабільнішим (підказка: перехідні сезони зазвичай нестійкіші за глибоку зиму чи глибоке літо).

In [3]:
# Групування за місяцем із розрахунком середнього та стандартного відхилення
monthly_stats = climate.groupby("місяць")["температура"].agg(["mean", "std"])
monthly_stats.columns = ["середня_темп", "стандартне_відхилення"]

max_std_month = monthly_stats["стандартне_відхилення"].idxmax()
max_std_value = monthly_stats["стандартне_відхилення"].max()

print(f"Найбільший розкид (std) спостерігається у місяці {max_std_month} зі значенням {max_std_value:.4f} °C")
monthly_stats

Найбільший розкид (std) спостерігається у місяці 8 зі значенням 1.3817 °C


,середня_темп,стандартне_відхилення
місяць,,
1,-3.400,0.424264
2,-2.875,1.129528
3,1.600,1.023067
4,8.875,0.865544
5,14.225,0.727438
6,18.900,0.294392
7,20.500,1.116542
8,19.125,1.381726
9,14.375,1.250000


### Письмовий висновок до Завдання 2:
- **Місяць із найбільшим розкидом:** Найбільше стандартне відхилення зафіксовано у **місяці 8 (серпень)** — $\text{std} \approx 1.3817$ °C (також високий розкид спостерігається у вересні: 1.2500 °C, грудні: 1.1343 °C та лютому: 1.1295 °C).
- **Припущення щодо причин нестабільності:**
  1. *З кліматичного погляду:* Кінець літа та перехід до осені (серпень-вересень), як і перехідні місяці загалом, характеризуються зміною масштабних атмосферних циркуляцій. У ці періоди в помірних широтах (зокрема у Львові) погода значно варіюється між роками: в один рік серпень може бути продовженням тривалої спекотної антициклональної погоди, а в інший — супроводжуватися раннім надходженням прохолодних атлантичних повітряних мас і дощів. Така сама нестабільність притаманна зимовим перехідним місяцям (грудень, лютий) через чергування відлиг і морозів.
  2. *З погляду моделювання:* Оскільки генерація здійснювалася за вибіркою лише з 4 років ($n=4$), стандартне відхилення вибірки чутливе до конкретних реалізацій нормального шуму (з seed=42 у серпні 2022 було 17.5 °C, а у 2023 — 20.7 °C, що створило найбільший розмах коливань у 3.2 °C).

## Завдання 3. pivot_table

**Завдання:** Побудуйте зведену таблицю `pivot_table()` із місяцями (1–12) як індексом, роками як стовпцями і середньою температурою як значенням — класична таблиця "місяці в рядках, роки в стовпцях". Порівняйте її візуально з вихідним довгим `climate` і письмово поясніть, яка з двох форм (довга чи ця зведена) зручніша для читання людиною, а яка — для подальшого групування чи побудови графіка (зв'язок із Лекцією 4: melt/pivot).

In [4]:
# Побудова зведеної таблиці: місяці в індексі, роки у стовпцях
pivot_climate = climate.pivot_table(index="місяць", columns="рік", values="температура", aggfunc="mean")
pivot_climate

рік,2021,2022,2023,2024
місяць,,,,
1,-3.0,-3.3,-4.0,-3.3
2,-2.0,-3.8,-1.8,-3.9
3,3.1,0.8,1.3,1.2
4,10.0,7.9,8.9,8.7
5,14.3,13.5,13.9,15.2
6,18.7,19.2,18.6,19.1
7,22.1,19.6,19.9,20.4
8,19.7,17.5,20.7,18.6
9,14.0,16.0,14.5,13.0


### Письмовий висновок до Завдання 3:
- **Порівняння форм представлення даних:**
  1. **Зведена (широка, wide) форма (`pivot_table`):**
     - *Призначення:* Найзручніша **для візуального сприйняття людиною**.
     - *Переваги:* Дозволяє в компактному двовимірному вигляді одночасно охопити всю динаміку: по рядках легко простежити річний хід температури (сезонність від січня до грудня), а по стовпцях — миттєво зіставити один і той самий місяць у різні роки (наприклад, порівняти липень 2021 року: 22.1 °C та липень 2022 року: 19.6 °C).
  2. **Довга (tidy/long) форма (`climate`):**
     - *Призначення:* Найзручніша **для подальшого групування (`groupby`), фільтрації, моделювання та побудови графіків** (у `matplotlib`, `seaborn` тощо).
     - *Переваги:* Кожен рядок є окремим атомарним спостереженням, а кожен стовпець («місто», «рік», «місяць», «температура») є окремою явною змінною. До неї легко застосовувати векторизовані операції, додавати нові виміри, фільтрувати чи передавати як параметри `x="місяць"`, `y="температура"`, `hue="рік"` у бібліотеки візуалізації.
- **Зв'язок із Лекцією 4 (tidy data):**
  - Перетворення з довгого формату в широкий здійснюється операціями `pivot()` / `pivot_table()` (для звітів, презентацій та читання людиною).
  - Зворотне перетворення з широкого формату в довгий tidy-формат здійснюється операцією `melt()` (для приведення сирих таблиць до аналітичного стандарту).

## Завдання 4. Похідні категорії і crosstab

**Завдання:** Додайте до `climate` два похідні категоріальні стовпці: "сезон" (зима/весна/літо/осінь за номером місяця) і "тепліше_за_середнє" (булеве значення: чи температура цього рядка вища за середньорічну температуру вашого варіанта — 8.5 °C). Побудуйте `pd.crosstab()` цих двох стовпців і письмово перевірте, чи розподіл частот узгоджується з очікуваннями (наприклад, чи всі літні місяці справді потрапили в категорію "тепліше за середнє").

In [5]:
# Функція для визначення сезону за номером місяця
def get_season(month):
    if month in [12, 1, 2]:
        return "зима"
    elif month in [3, 4, 5]:
        return "весна"
    elif month in [6, 7, 8]:
        return "літо"
    else:
        return "осінь"

# Додавання похідних стовпців
climate["сезон"] = climate["місяць"].apply(get_season)
climate["тепліше_за_середнє"] = climate["температура"] > base_temp

# Побудова таблиці спряженості (crosstab)
season_ct = pd.crosstab(climate["сезон"], climate["тепліше_за_середнє"], margins=True, margins_name="Всього")
season_ct

тепліше_за_середнє,False,True,Всього
сезон,,,
весна,5,7,12
зима,12,0,12
літо,0,12,12
осінь,7,5,12
Всього,24,24,48


### Письмовий висновок до Завдання 4:
- **Отриманий розподіл частот:**
  - **Зима (12 спостережень):** 12 разів `False` (0 разів `True`) — 100% зимових місяців холодніші за 8.5 °C.
  - **Літо (12 спостережень):** 0 разів `False` (12 разів `True`) — 100% літніх місяців тепліші за 8.5 °C.
  - **Весна (12 спостережень):** 5 разів `False`, 7 разів `True`.
  - **Осінь (12 спостережень):** 7 разів `False`, 5 разів `True`.
- **Узгодженість із очікуваннями:**
  Розподіл частот **повністю узгоджується з фізичними та кліматичними очікуваннями**:
  1. Усі екстремальні сезони (зима і літо) демонструють 100%-ву визначеність: влітку температура гарантовано вища за середньорічну (середнє липня сягає 20.5 °C при амплітуді 12 °C), а взимку — гарантовано нижча (середнє січня становить -3.4 °C).
  2. У перехідних сезонах (весна й осінь) відбувається перехід через середньорічну позначку 8.5 °C. Наприклад, у березні температура ще низька (~1.6 °C), у квітні близька до середньої (~8.9 °C), а в травні вже суттєво вища (~14.2 °C). Аналогічно восени: вересень ще теплий (~14.4 °C), жовтень на межі (~8.1 °C), а листопад уже холодний (~2.5 °C).

## Завдання 5. pivot() на своєму наборі

**Завдання:** Спробуйте `pivot()` (без aggfunc) на своєму `climate` з індексом "місяць" і стовпцями "рік". Перевірте, чи він спрацьовує (на відміну від прикладу з кав'ярнями), і письмово поясніть чому — з посиланням на кількість рядків, що припадає на кожну комбінацію місяць/рік у вашому наборі. Опишіть, яку зміну в структурі даних довелося б внести (наприклад, додати кілька вимірювальних станцій на місто), щоб той самий `pivot()` почав завершуватися помилкою, як у прикладі з кав'ярнями.

In [6]:
# Спроба виклику pivot() без агрегувальної функції
pivot_simple = climate.pivot(index="місяць", columns="рік", values="температура")
pivot_simple

рік,2021,2022,2023,2024
місяць,,,,
1,-3.0,-3.3,-4.0,-3.3
2,-2.0,-3.8,-1.8,-3.9
3,3.1,0.8,1.3,1.2
4,10.0,7.9,8.9,8.7
5,14.3,13.5,13.9,15.2
6,18.7,19.2,18.6,19.1
7,22.1,19.6,19.9,20.4
8,19.7,17.5,20.7,18.6
9,14.0,16.0,14.5,13.0


In [7]:
# Перевірка кількості рядків на пару (місяць, рік)
counts_per_pair = climate.groupby(["місяць", "рік"]).size()
print(f"Максимальна кількість рядків на комбінацію (місяць, рік): {counts_per_pair.max()}")
print(f"Мінімальна кількість рядків на комбінацію (місяць, рік): {counts_per_pair.min()}")
print(f"Всього унікальних комбінацій: {len(counts_per_pair)} з {len(climate)} рядків датасету")

# Демонстрація: зміна структури даних (додавання метеостанцій), що ламає pivot()
climate_multistation = pd.concat([
    climate.assign(станція="Львів-Аеропорт"),
    climate.assign(станція="Львів-Погулянка")
], ignore_index=True)

print("\nСпроба виконати pivot() на даних із двома станціями:")
try:
    climate_multistation.pivot(index="місяць", columns="рік", values="температура")
except ValueError as err:
    print("Помилка ValueError:", err)

Максимальна кількість рядків на комбінацію (місяць, рік): 1
Мінімальна кількість рядків на комбінацію (місяць, рік): 1
Всього унікальних комбінацій: 48 з 48 рядків датасету

Спроба виконати pivot() на даних із двома станціями:
Помилка ValueError: Index contains duplicate entries, cannot reshape


### Письмовий висновок до Завдання 5:
1. **Чому `pivot()` успішно спрацював на нашому наборі?**
   Метод `pivot()` призначений виключно для реструктуризації форми даних без агрегації. Він вимагає, щоб кожна пара комбінації `(index, columns)` зустрічалася в таблиці **рівно один раз**. У нашому датасеті `climate` є 48 рядків, що відповідають 4 рокам та 12 місяцям: для кожної комбінації `(місяць, рік)` припадає **рівно один рядок (одне значення температури)**. Відповідно, немає неоднозначності, яке значення покласти у відповідну комірку зведеної таблиці, тому `pivot()` успішно відпрацював.
2. **Чому `pivot()` ламався у прикладі з кав'ярнями?**
   У прикладі з кав'ярнями на пару `(філія, рік)` припадало 4 різні рядки (по одному на кожен квартал року). Метод `pivot()` не знає, що робити з чотирма числами без явного правила зведення, тому викидає `ValueError: Index contains duplicate entries, cannot reshape`.
3. **Яку зміну в структурі даних треба внести, щоб зламати `pivot()` у кліматичному наборі?**
   Щоб `pivot()` завершився аналогічною помилкою `ValueError`, достатньо порушити унікальність комбінації `(місяць, рік)`. Наприклад:
   - Додати вимірювання з **кількох різних метеостанцій міста** (як продемонстровано вище в коді: станція "Львів-Аеропорт" та станція "Львів-Погулянка");
   - Додати часову дискретизацію вищої частоти: проводити вимірювання за декадами місяця (1-ша, 2-га, 3-тя декади) або щодня;
   - Додати вимірювання у різний час доби (день / ніч).
   У будь-якому з цих випадків на одну комірку `(місяць, рік)` припадатиме більше одного рядка, тому для побудови зведеної таблиці стане обов'язковим використання `pivot_table(..., aggfunc=...)`.

---
## Відповіді на контрольні запитання

### 1. Чи можете пояснити словами логіку split-apply-combine, яку виконує `groupby()` — що саме відбувається на кожному з трьох кроків?

Концепція **Split-Apply-Combine** (розбиття – застосування – комбінування) є фундаментальним патерном обробки даних у бібліотеці `pandas`:
1. **Split (Розбиття):** Початковий датафрейм розділяється на незалежні групи/підвибірки за унікальними значеннями зазначеного ключа (або комбінації кількох ключів). При цьому фізично дані можуть не копіюватися: створюється відображення індексів рядків для кожної сформованої групи.
2. **Apply (Застосування):** До кожної виокремленої групи незалежно застосовується зазначена функція. Це може бути:
   - *Агрегація* (`agg`, `mean`, `sum`, `std`) — перетворення групи значень у єдине підсумкове скалярне значення;
   - *Трансформація* (`transform`) — обчислення, що повертає об'єкт тієї самої розмірності, що й вихідна група (наприклад, стандартизація значень у межах групи);
   - *Фільтрація* (`filter`) — відбір лише тих груп, які відповідають певному груповому критерію (наприклад, залишати групи, де кількість спостережень $> 10$).
3. **Combine (Комбінування / Об'єднання):** Результати застосування функції до кожної окремої групи збираються назад у єдиний результуючий об'єкт (`Series` або `DataFrame`), де ключі групування формують новий індекс результату.

---

### 2. Чи розумієте принципову відмінність між `pivot()` і `pivot_table()` — чому один завершується помилкою на дублікатах комбінацій індекс/стовпець, а другий — ні?

- **`pivot()`** — це низькорівнева операція зміни структури форми таблиці (чисте транспонування/розгортання без обчислень). Вона очікує бієктивну відповідність: кожна комірка на перетині `(index, columns)` повинна однозначно заповнюватися рівно одним значенням із `values`. Якщо на якусь пару припадає 2 або більше рядків, алгоритм не має інструкцій, яке значення обрати, і тому генерує виняток `ValueError: Index contains duplicate entries`.
- **`pivot_table()`** — це інструмент побудови повноцінної зведеної аналітичної таблиці, яка **включає обов'язковий механізм агрегації** (параметр `aggfunc`, за замовчуванням `'mean'`). Якщо для певної комбінації `(index, columns)` виявлено кілька спостережень, `pivot_table()` не викидає помилку, а агрегує ці спостереження в єдине значення за допомогою заданої функції (середнє, сума, медіана тощо).

---

### 3. Чи можете пояснити, що показує `crosstab()` і чим він відрізняється від еквівалентного результату через `groupby(...).size()`?

- **Що показує `crosstab()`:** Функція `pd.crosstab()` обчислює **таблицю спряженості (частотний розподіл)** між двома або більше категоріальними факторами. Вона показує, скільки разів (або з якою відносною часткою/ймовірністю, якщо ввімкнено `normalize`) кожна комбінація категорій зустрілася у спостереженнях.
- **Відмінність від `groupby(...).size()`:**
  1. *Формат виводу:* `df.groupby(["A", "B"]).size()` повертає одновимірний об'єкт `Series` із дворівневим `MultiIndex` (довгий формат). Для перетворення його у зручну матрицю необхідно додатково викликати `.unstack(fill_value=0)`. Натомість `crosstab()` одразу повертає готову двовимірну прямокутну таблицю `DataFrame` (широкий формат).
  2. *Зручність параметрів:* `crosstab()` має вбудовану підтримку підсумкових сум по рядках і стовпцях (`margins=True`, `margins_name="Total"`) та гнучку нормалізацію частот (`normalize='all'`, `normalize='index'`, `normalize='columns'`), що у випадку `groupby` вимагало б написання громіздких додаткових кроків обчислень.

---

### 4. Чи можете пояснити, чому дані для `groupby` і `pivot_table` мають бути в довгому (tidy) форматі, а не в таблиці з роками чи місяцями, розтягнутими в назви стовпців?

- У **довгому (tidy) форматі** діє правило: *кожна змінна — це окремий стовпець, кожне спостереження — окремий рядок, кожен тип спостережної одиниці — окрема таблиця*. Тобто фактори вимірювання («місяць», «рік», «місто») зберігаються як звичайні значення даних у відповідних стовпцях-вимірах, а виміряна величина («температура») — у цільовому стовпці значень.
- **Чому це критично для `groupby` і `pivot_table`:**
  1. *Універсальність агрегацій:* Методи `groupby(by="рік")` та `pivot_table(index="місяць", columns="рік")` очікують назву стовпця як ключ для групування. Якщо роки розміщені в заголовках стовпців (широкий формат, де стовпці: `2021`, `2022`, `2023`, `2024`), ми не можемо згрупувати дані за категорією «рік», оскільки самої змінної «рік» як стовпця не існує в датафреймі.
  2. *Гнучкість аналізу та візуалізації:* У довгому форматі легко додавати нові спостереження (нові роки чи місяці) простим додаванням рядків без зміни схеми таблиці (без додавання нових стовпців). Крім того, всі сучасні бібліотеки візуалізації (`seaborn`, `plotly`) та статистичні моделі оптимізовані саме під tidy data.